# Day 4: Performance Analytics
This notebook performs advanced performance analytics on the mutual fund datasets, including CAGR, Sharpe/Sortino ratios, Alpha/Beta calculation, and a comprehensive Fund Scorecard.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
from scipy import stats
import os

# Settings
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Paths
db_path = r'../../bluestock_mf.db'
report_dir = r'../../reports/analytics'
if not os.path.exists(report_dir):
    os.makedirs(report_dir)

engine = create_engine(f'sqlite:///{db_path}')
print("Setup Complete!")

## 1. Daily Returns
Calculating `daily_return = nav_t / nav_t-1 - 1` for all 40 schemes.

In [ ]:
query_nav = """
SELECT n.date, n.nav, f.scheme_name, f.amfi_code
FROM fact_nav n
JOIN dim_fund f ON n.amfi_code = f.amfi_code
ORDER BY f.amfi_code, n.date
"""
df_nav = pd.read_sql(query_nav, engine)
df_nav['date'] = pd.to_datetime(df_nav['date'])

# Calculate Daily Returns per AMFI code
df_nav['daily_return'] = df_nav.groupby('amfi_code')['nav'].pct_change()

# Validate distribution
plt.figure(figsize=(10, 6))
sns.histplot(df_nav['daily_return'].dropna(), bins=100, kde=True, color='blue')
plt.title('Distribution of Daily Returns (All Schemes)')
plt.xlabel('Daily Return')
plt.show()

## 2. CAGR Calculation (1yr, 3yr, 5yr)
`CAGR = (NAV_end / NAV_start) ^ (1/n) - 1`

In [ ]:
def calculate_cagr(df_group, years):
    df_group = df_group.sort_values('date')
    end_nav = df_group['nav'].iloc[-1]
    # Approximate start NAV based on years
    days = int(years * 365)
    if len(df_group) < days:
        return np.nan
    start_nav = df_group['nav'].iloc[-days]
    return (end_nav / start_nav)**(1/years) - 1

cagr_results = df_nav.groupby(['amfi_code', 'scheme_name']).apply(lambda x: pd.Series({
    'cagr_1yr': calculate_cagr(x, 1),
    'cagr_3yr': calculate_cagr(x, 3),
    'cagr_5yr': calculate_cagr(x, 5)
})).reset_index()

cagr_results.head()

## 3. Sharpe & Sortino Ratios
- Sharpe: `(Rp - Rf) / Std(Rp) * sqrt(252)`
- Sortino: `(Rp - Rf) / DownsideStd(Rp) * sqrt(252)`
- Risk-free rate (Rf) = 6.5%

In [ ]:
rf_daily = 0.065 / 252

def calculate_ratios(group):
    returns = group['daily_return'].dropna()
    excess_returns = returns - rf_daily
    
    # Sharpe
    sharpe = (excess_returns.mean() / returns.std()) * np.sqrt(252)
    
    # Sortino
    downside_returns = returns[returns < 0]
    sortino = (excess_returns.mean() / downside_returns.std()) * np.sqrt(252)
    
    return pd.Series({'sharpe_ratio': sharpe, 'sortino_ratio': sortino})

ratios_df = df_nav.groupby(['amfi_code', 'scheme_name']).apply(calculate_ratios).reset_index()
ratios_df.head()

## 4. Alpha and Beta (vs Nifty 100)
Using OLS regression: `Fund_Return = Alpha + Beta * Benchmark_Return`

In [ ]:
# Get Nifty 100 returns
query_benchmark = "SELECT date, close_value FROM fact_benchmark_indices WHERE index_name = 'NIFTY100' ORDER BY date"
df_bench = pd.read_sql(query_benchmark, engine)
df_bench['date'] = pd.to_datetime(df_bench['date'])
df_bench['bench_return'] = df_bench['close_value'].pct_change()

alpha_beta_results = []

for amfi in df_nav['amfi_code'].unique():
    fund_data = df_nav[df_nav['amfi_code'] == amfi][['date', 'daily_return', 'scheme_name']]
    merged = pd.merge(fund_data, df_bench, on='date').dropna()
    
    if len(merged) > 30:
        slope, intercept, r_value, p_value, std_err = stats.linregress(merged['bench_return'], merged['daily_return'])
        alpha_beta_results.append({
            'amfi_code': amfi,
            'scheme_name': merged['scheme_name'].iloc[0],
            'beta': slope,
            'alpha_annualized': intercept * 252,
            'r_squared': r_value**2
        })

df_alpha_beta = pd.DataFrame(alpha_beta_results)
df_alpha_beta.to_csv(os.path.join(report_dir, 'alpha_beta.csv'), index=False)
df_alpha_beta.head()

## 5. Maximum Drawdown
`MDD = min(NAV / running_max - 1)`

In [ ]:
def calculate_mdd(group):
    nav = group['nav']
    running_max = nav.cummax()
    drawdown = (nav / running_max) - 1
    max_drawdown = drawdown.min()
    
    # Find dates
    mdd_idx = drawdown.idxmin()
    peak_idx = nav[:mdd_idx].idxmax()
    
    return pd.Series({
        'max_drawdown': max_drawdown,
        'mdd_start': group.loc[peak_idx, 'date'],
        'mdd_end': group.loc[mdd_idx, 'date']
    })

mdd_df = df_nav.groupby(['amfi_code', 'scheme_name']).apply(calculate_mdd).reset_index()
mdd_df.head()

## 6. Fund Scorecard (0-100)
Weights:
- 30% × 3yr return rank
- 25% × Sharpe rank
- 20% × Alpha rank
- 15% × Expense ratio rank (inverse)
- 10% × Max DD rank (inverse)

In [ ]:
# Get expense ratios
query_exp = "SELECT amfi_code, expense_ratio_pct FROM fact_performance"
df_exp = pd.read_sql(query_exp, engine)

# Merge all metrics
scorecard_df = pd.merge(cagr_results, ratios_df, on=['amfi_code', 'scheme_name'])
scorecard_df = pd.merge(scorecard_df, df_alpha_beta, on=['amfi_code', 'scheme_name'])
scorecard_df = pd.merge(scorecard_df, mdd_df, on=['amfi_code', 'scheme_name'])
scorecard_df = pd.merge(scorecard_df, df_exp, on='amfi_code')

# Calculate percentile ranks (0 to 1)
scorecard_df['rank_3yr'] = scorecard_df['cagr_3yr'].rank(pct=True)
scorecard_df['rank_sharpe'] = scorecard_df['sharpe_ratio'].rank(pct=True)
scorecard_df['rank_alpha'] = scorecard_df['alpha_annualized'].rank(pct=True)
scorecard_df['rank_expense'] = scorecard_df['expense_ratio_pct'].rank(pct=True, ascending=False)
scorecard_df['rank_mdd'] = scorecard_df['max_drawdown'].rank(pct=True) # Higher drawdown is worse (more negative)

# Composite Score
scorecard_df['final_score'] = (
    0.30 * scorecard_df['rank_3yr'] +
    0.25 * scorecard_df['rank_sharpe'] +
    0.20 * scorecard_df['rank_alpha'] +
    0.15 * scorecard_df['rank_expense'] +
    0.10 * scorecard_df['rank_mdd']
) * 100

scorecard_df = scorecard_df.sort_values('final_score', ascending=False)
scorecard_df.to_csv(os.path.join(report_dir, 'fund_scorecard.csv'), index=False)
scorecard_df[['scheme_name', 'final_score']].head(10)

## 7. Benchmark Comparison Chart
Top 5 funds vs Nifty 50 and Nifty 100.

In [ ]:
top_5_amfi = scorecard_df.head(5)['amfi_code'].tolist()

# Get benchmark data
query_bench_all = "SELECT date, index_name, close_value FROM fact_benchmark_indices WHERE index_name IN ('NIFTY50', 'NIFTY100')"
df_bench_all = pd.read_sql(query_bench_all, engine)
df_bench_all['date'] = pd.to_datetime(df_bench_all['date'])

# Pivot benchmarks
bench_pivot = df_bench_all.pivot(index='date', columns='index_name', values='close_value')
bench_normalized = bench_pivot / bench_pivot.iloc[0] * 100

# Get Top 5 NAVs
top_5_nav = df_nav[df_nav['amfi_code'].isin(top_5_amfi)].pivot(index='date', columns='scheme_name', values='nav')
top_5_normalized = top_5_nav / top_5_nav.iloc[0] * 100

# Combine and Plot
plt.figure(figsize=(14, 8))
for col in top_5_normalized.columns:
    plt.plot(top_5_normalized.index, top_5_normalized[col], label=col, alpha=0.7)
plt.plot(bench_normalized.index, bench_normalized['NIFTY50'], label='Nifty 50', color='black', linewidth=2, linestyle='--')
plt.plot(bench_normalized.index, bench_normalized['NIFTY100'], label='Nifty 100', color='red', linewidth=2, linestyle='--')

plt.title('Top 5 Funds vs Benchmarks (Normalized)')
plt.ylabel('Normalized Value (Base 100)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(report_dir, 'benchmark_comparison.png'))
plt.show()

# Tracking Error
tracking_errors = []
nifty_100_returns = bench_pivot['NIFTY100'].pct_change().dropna()
for amfi in top_5_amfi:
    fund_ret = df_nav[df_nav['amfi_code'] == amfi].set_index('date')['daily_return'].dropna()
    merged = pd.concat([fund_ret, nifty_100_returns], axis=1).dropna()
    te = (merged['daily_return'] - merged['NIFTY100']).std() * np.sqrt(252)
    tracking_errors.append({'scheme': scorecard_df[scorecard_df['amfi_code']==amfi]['scheme_name'].iloc[0], 'tracking_error': te})

pd.DataFrame(tracking_errors)